# CNN on log-mel-spectrograms — Strategy A

Trains a small 4-block 2-D CNN (~390K params) on per-segment
log-mel-spectrograms (1 × 128 × 100). Same train/val/test split as
every other model in this project (`splits.npz`, seed=9).

**Goal of this experiment:** test whether a representation-learning
model on raw spectrograms can beat the aggregated-feature ceiling
hit by LogReg / RF / XGB / MLP (~0.55 macro F1 on test).


## Cell 1 — Setup


In [1]:
# ============================================================================
# Cell 1 — Setup: imports, paths, device, reproducibility
# ============================================================================
import sys, time, json
from pathlib import Path

import numpy as np
import torch

# Resolve project root (contains config.json + data/)
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'config.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / '_shared'))

from data       import load_splits
from melspec    import load_cache, MelSpecDataset
from cnn        import MelCNN, count_params
from train_nn   import auto_device, eval_probs, mean_ap, train_one_run, seed_all
from imbalance  import compute_class_weights, make_focal_loss
from eval       import (macro_f1, tune_thresholds_on_val,
                        predict_with_threshold, classification_report)

STRATEGY   = 'A'
NB_ID      = '08'
MODEL_NAME = 'cnn'
SEED       = 42

DEVICE = auto_device()
print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
seed_all(SEED)


Project root: /Users/amrabdullah/AI_2026ss/mlpc
Device:       mps


## Cell 2 — Load splits + mel-spec cache


In [2]:
# ============================================================================
# Cell 2 — Load data, build DataLoaders
# ============================================================================
splits = load_splits(STRATEGY)
print(splits.summary())

cache, meta = load_cache()
print(f"\nMel-spec cache: {cache.shape} {cache.dtype}")
print(f"  global stats: mean={float(meta['global_mean']):.3f} "
      f"std={float(meta['global_std']):.3f}")

# Datasets share the memmap; no extra RAM
ds_train = MelSpecDataset(splits.train_idx, splits.Y_train)
ds_val   = MelSpecDataset(splits.val_idx,   splits.Y_val)
ds_test  = MelSpecDataset(splits.test_idx,  splits.Y_test)

from torch.utils.data import DataLoader
BATCH = 128
N_WORKERS = 0   # memmap + macOS works best with 0 workers

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                      num_workers=N_WORKERS, drop_last=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH * 2, shuffle=False,
                      num_workers=N_WORKERS)
dl_test  = DataLoader(ds_test,  batch_size=BATCH * 2, shuffle=False,
                      num_workers=N_WORKERS)

print(f'\nBatches: train={len(dl_train)}, val={len(dl_val)}, test={len(dl_test)}')


Strategy A
  X_train: (100166, 960)  Y_train: (100166, 15)
  X_val:   (34245, 960)  Y_val:   (34245, 15)
  X_test:  (33828, 960)  Y_test:  (33828, 15)
  D = 960 features, C = 15 classes

Mel-spec cache: (168239, 128, 100) float16
  global stats: mean=4.531 std=2.018

Batches: train=782, val=134, test=133


## Cell 3 — Model factory & sanity check
`fresh_model()` re-seeds before instantiating to guarantee that
each config starts from the same initial weights.


In [3]:
# ============================================================================
# Cell 3 — Model factory
# ============================================================================
def fresh_model():
    seed_all(SEED)
    return MelCNN(n_classes=int(splits.C), dropout=0.4, base_ch=32)

_m = fresh_model()
print(f'MelCNN params: {count_params(_m):,}')
with torch.no_grad():
    _x = torch.randn(2, 1, 128, 100)
    _y = _m(_x)
print(f'shape check: {tuple(_x.shape)} -> {tuple(_y.shape)}')
del _m, _x, _y


MelCNN params: 392,175
shape check: (2, 1, 128, 100) -> (2, 15)


## Cell 4 — Config 1: vanilla BCE
Uncalibrated baseline. Early-stop on val mean-AP (threshold-free,
the right metric for under-trained calibrated probabilities — see
MLP v1 retro in `terminal_claude.md`).


In [4]:
# ============================================================================
# Cell 4 — Config 1: vanilla BCE
# ============================================================================
print('=== Config 1: vanilla BCE ===')
m1 = fresh_model().to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(m1.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state1, hist1 = train_one_run(
    m1, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C1 done in {time.time() - t0:.1f}s | {hist1.summary()}')

m1.load_state_dict(state1)
probs_val_c1, Y_val = eval_probs(m1, dl_val, DEVICE)
print(f'C1 val: mAP={mean_ap(probs_val_c1, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c1>=0.5).astype(np.int8)):.4f}")


=== Config 1: vanilla BCE ===


  ep  1  tr_loss=-462641820274296815616.0000  val_mAP=0.4860  val_F1@0.5=0.3757  skipped=0


  ep  2  tr_loss=0.1031  val_mAP=0.5265  val_F1@0.5=0.3830  skipped=34


  ep  3  tr_loss=0.0958  val_mAP=0.5377  val_F1@0.5=0.3816  skipped=2


  ep  4  tr_loss=0.0910  val_mAP=0.5611  val_F1@0.5=0.4304  skipped=2


...

  ep 16  tr_loss=0.0627  val_mAP=0.5984  val_F1@0.5=0.4867  skipped=0


  ep 17  tr_loss=0.0611  val_mAP=0.5914  val_F1@0.5=0.4570  skipped=0


  ep 18  tr_loss=0.0598  val_mAP=0.5984  val_F1@0.5=0.4984  skipped=2


  ep 19  tr_loss=0.0581  val_mAP=0.5969  val_F1@0.5=0.4939  skipped=0
  early stop at epoch 19 (best=13)
C1 done in 7135.5s | best epoch=13, best val mAP=0.5999, final val F1@0.5=0.4939, final train loss=0.0581


C1 val: mAP=0.5999 F1@0.5=0.5049


## Cell 5 — Config 2: pos-weighted BCE
`sqrt_inverse_freq` clipped to [0.5, 30]. The aggressive
`balanced` weights for `light_switch` (≈370×) destabilize CNN
training; clipping is essential.


In [5]:
# ============================================================================
# Cell 5 — Config 2: pos-weighted BCE (sqrt_inverse_freq)
# ============================================================================
print('=== Config 2: pos-weighted BCE ===')
pw = compute_class_weights(splits.Y_train,
                           kind='sqrt_inverse_freq', clip=(0.5, 30.0))
print(f'pos_weight: min={pw.min():.2f}  max={pw.max():.2f}')

m2 = fresh_model().to(DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(pw, device=DEVICE))
opt = torch.optim.AdamW(m2.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state2, hist2 = train_one_run(
    m2, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C2 done in {time.time() - t0:.1f}s | {hist2.summary()}')

m2.load_state_dict(state2)
probs_val_c2, _ = eval_probs(m2, dl_val, DEVICE)
print(f'C2 val: mAP={mean_ap(probs_val_c2, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c2>=0.5).astype(np.int8)):.4f}")


=== Config 2: pos-weighted BCE ===
pos_weight: min=2.79  max=19.15


  ep  1  tr_loss=0.3256  val_mAP=0.4862  val_F1@0.5=0.4482  skipped=0


  ep  2  tr_loss=0.2720  val_mAP=0.5337  val_F1@0.5=0.4770  skipped=1


  ep  3  tr_loss=-2414736090.6129  val_mAP=0.5528  val_F1@0.5=0.5200  skipped=3


  ep  4  tr_loss=0.2389  val_mAP=0.5642  val_F1@0.5=0.5121  skipped=0


...

  ep 29  tr_loss=0.1149  val_mAP=0.6120  val_F1@0.5=0.5894  skipped=1


  ep 30  tr_loss=0.1123  val_mAP=0.6109  val_F1@0.5=0.5934  skipped=1


  ep 31  tr_loss=0.1115  val_mAP=0.6092  val_F1@0.5=0.5909  skipped=1


  ep 32  tr_loss=0.1122  val_mAP=0.6085  val_F1@0.5=0.5849  skipped=9
  early stop at epoch 32 (best=26)
C2 done in 9868.2s | best epoch=26, best val mAP=0.6144, final val F1@0.5=0.5849, final train loss=0.1122


C2 val: mAP=0.6144 F1@0.5=0.5938


## Cell 6 — Config 3: focal loss γ=2
Down-weights easy negatives — useful when the negative class
dominates by 100× (the case for `light_switch`).


In [6]:
# ============================================================================
# Cell 6 — Config 3: focal loss γ=2
# ============================================================================
print('=== Config 3: focal loss γ=2 ===')
m3 = fresh_model().to(DEVICE)
loss_fn = make_focal_loss(gamma=2.0, pos_weight=None).to(DEVICE)
opt = torch.optim.AdamW(m3.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

t0 = time.time()
state3, hist3 = train_one_run(
    m3, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=40, min_epochs=10, patience=6)
print(f'C3 done in {time.time() - t0:.1f}s | {hist3.summary()}')

m3.load_state_dict(state3)
probs_val_c3, _ = eval_probs(m3, dl_val, DEVICE)
print(f'C3 val: mAP={mean_ap(probs_val_c3, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c3>=0.5).astype(np.int8)):.4f}")


=== Config 3: focal loss γ=2 ===


  ep  1  tr_loss=-51930028247.8192  val_mAP=0.4535  val_F1@0.5=0.3294  skipped=4


  ep  2  tr_loss=-9343273776.0109  val_mAP=0.5238  val_F1@0.5=0.3809  skipped=4


  ep  3  tr_loss=0.0277  val_mAP=0.5206  val_F1@0.5=0.3494  skipped=11


  ep  4  tr_loss=0.0261  val_mAP=0.5538  val_F1@0.5=0.4264  skipped=12


...

  ep 26  tr_loss=0.0139  val_mAP=0.6102  val_F1@0.5=0.5325  skipped=1


  ep 27  tr_loss=0.0136  val_mAP=0.6071  val_F1@0.5=0.5158  skipped=4


  ep 28  tr_loss=0.0134  val_mAP=0.6104  val_F1@0.5=0.5296  skipped=1


  ep 29  tr_loss=-2106850821.9173  val_mAP=0.6096  val_F1@0.5=0.5301  skipped=5
  early stop at epoch 29 (best=23)
C3 done in 8685.6s | best epoch=23, best val mAP=0.6120, final val F1@0.5=0.5301, final train loss=-2106850821.9173


C3 val: mAP=0.6120 F1@0.5=0.5159


## Cell 7 — Config 4: best base + per-class threshold tuning
Tie-break by val mAP (threshold-free), then tune per-class
thresholds on val. This isolates the *calibration* effect from
the *learning* effect.


In [7]:
# ============================================================================
# Cell 7 — Pick val-mAP best base, tune per-class thresholds on val
# ============================================================================
configs = {
    'C1': (probs_val_c1, hist1, state1, m1),
    'C2': (probs_val_c2, hist2, state2, m2),
    'C3': (probs_val_c3, hist3, state3, m3),
}
best_key = max(configs, key=lambda k: mean_ap(configs[k][0], Y_val))
print(f'Best base (val mAP): {best_key}')

probs_val_best, _, state_best, model_best = configs[best_key]
class_names = list(splits.class_names)

thr_df = tune_thresholds_on_val(
    probs_val_best, Y_val,
    grid=np.arange(0.05, 0.96, 0.01),
    class_names=class_names)
print(thr_df.round(3))
thresholds = thr_df['threshold'].values
val_macro_f1_c4 = macro_f1(
    Y_val, predict_with_threshold(probs_val_best, thresholds))
print(f'C4 val macro F1 (tuned): {val_macro_f1_c4:.4f}')


Best base (val mAP): C2


                            threshold  val_f1  val_precision  val_recall  \
class                                                                      
bell_ringing                     0.55   0.536          0.582       0.498   
coffee_machine                   0.92   0.724          0.884       0.612   
cutlery_dishes                   0.64   0.637          0.677       0.601   
door_open_close                  0.64   0.399          0.367       0.437   
footsteps                        0.53   0.598          0.568       0.631   
keyboard_typing                  0.62   0.800          0.836       0.766   
keychain                         0.64   0.647          0.640       0.654   
light_switch                     0.49   0.333          0.432       0.271   
microwave                        0.35   0.706          0.744       0.671   
phone_ringing                    0.52   0.740          0.755       0.724   
running_water                    0.67   0.771          0.778       0.765   
toilet_flush

## Cell 8 — Pick overall winner on val macro F1, test once
Test set is touched ONCE per model in this project. No peeking.


In [8]:
# ============================================================================
# Cell 8 — Pick val winner on macro F1, evaluate on test
# ============================================================================
val_macro = {
    'C1_vanilla':         macro_f1(Y_val, (probs_val_c1 >= 0.5).astype(np.int8)),
    'C2_pos_weighted':    macro_f1(Y_val, (probs_val_c2 >= 0.5).astype(np.int8)),
    'C3_focal':           macro_f1(Y_val, (probs_val_c3 >= 0.5).astype(np.int8)),
    f'C4_{best_key}_thr': val_macro_f1_c4,
}
for k, v in val_macro.items():
    print(f'  {k:25s} {v:.4f}')

winner = max(val_macro, key=val_macro.get)
print(f'\nVal winner: {winner}')

if winner.startswith('C4'):
    probs_test, Y_test = eval_probs(model_best, dl_test, DEVICE)
    Y_pred_test = predict_with_threshold(probs_test, thresholds)
else:
    mdl = {'C1_vanilla': m1, 'C2_pos_weighted': m2, 'C3_focal': m3}[winner]
    probs_test, Y_test = eval_probs(mdl, dl_test, DEVICE)
    Y_pred_test = (probs_test >= 0.5).astype(np.int8)

test_report = classification_report(Y_test, Y_pred_test,
                                   class_names=class_names)
test_macro    = float(test_report.loc['macro avg',    'f1'])
test_micro    = float(test_report.loc['micro avg',    'f1'])
test_weighted = float(test_report.loc['weighted avg', 'f1'])
print(f'\nTEST macro F1:    {test_macro:.4f}')
print(f'TEST micro F1:    {test_micro:.4f}')
print(f'TEST weighted F1: {test_weighted:.4f}')


  C1_vanilla                0.5049
  C2_pos_weighted           0.5938
  C3_focal                  0.5159
  C4_C2_thr                 0.6066

Val winner: C4_C2_thr



TEST macro F1:    0.6035
TEST micro F1:    0.6881
TEST weighted F1: 0.6895


## Cell 9 — Plots


In [9]:
# ============================================================================
# Cell 9 — Training curves + per-class F1 + confusion matrix
# ============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

RESULTS = PROJECT_ROOT / 'strategy_A_baseline' / 'artifacts' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Training curves -- one figure per config
for cfg_name, hist in [('c1', hist1), ('c2', hist2), ('c3', hist3)]:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.train_loss)
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('train loss')
    ax[0].set_title(f'08 CNN {cfg_name.upper()} — train loss')
    ax[1].plot(hist.val_mAP,   label='val mAP')
    ax[1].plot(hist.val_f1_05, label='val F1@0.5')
    ax[1].axvline(hist.best_epoch - 1, ls='--', c='gray', alpha=0.5,
                  label=f'best ep={hist.best_epoch}')
    ax[1].set_xlabel('epoch'); ax[1].legend()
    ax[1].set_title(f'08 CNN {cfg_name.upper()} — val metrics')
    plt.tight_layout()
    plt.savefig(RESULTS / f'08_cnn_curves_{cfg_name}.png',
                dpi=120, bbox_inches='tight')
    plt.close()

# Per-class F1 bar chart (test)
fig, ax = plt.subplots(figsize=(10, 4))
cls = class_names
ax.bar(range(len(cls)), test_report.loc[cls, 'f1'].values)
ax.set_xticks(range(len(cls))); ax.set_xticklabels(cls, rotation=45, ha='right')
ax.set_ylabel('F1 (test)')
ax.set_title(f'08 CNN ({winner}) — per-class F1 on test')
plt.tight_layout()
plt.savefig(RESULTS / '08_cnn_per_class_f1.png',
            dpi=120, bbox_inches='tight')
plt.close()

# Confusion matrix (argmax single-label)
Y_t = Y_test.argmax(axis=1); Y_p = Y_pred_test.argmax(axis=1)
cm = confusion_matrix(Y_t, Y_p, labels=range(splits.C))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(splits.C)); ax.set_yticks(range(splits.C))
ax.set_xticklabels(cls, rotation=45, ha='right'); ax.set_yticklabels(cls)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'08 CNN ({winner}) — confusion (argmax)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(RESULTS / '08_cnn_confusion.png',
            dpi=120, bbox_inches='tight')
plt.close()

print(f'plots -> {RESULTS}')


plots -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results


## Cell 10 — Save results JSON


In [10]:
# ============================================================================
# Cell 10 — Save 08_cnn.json (schema mirrors 07_mlp.json)
# ============================================================================
result = {
    'meta': {
        'strategy':    STRATEGY,
        'model':       MODEL_NAME,
        'notebook_id': NB_ID,
        'n_train':     int(splits.N_train),
        'n_val':       int(splits.N_val),
        'n_test':      int(splits.N_test),
        'n_params':    int(count_params(m1)),
        'device':      str(DEVICE),
        'random_seed': SEED,
    },
    'val_scores_macro_f1': val_macro,
    'val_scores_mAP': {
        'C1': float(mean_ap(probs_val_c1, Y_val)),
        'C2': float(mean_ap(probs_val_c2, Y_val)),
        'C3': float(mean_ap(probs_val_c3, Y_val)),
    },
    'histories': {
        'C1': {'train_loss': hist1.train_loss, 'val_mAP': hist1.val_mAP,
               'val_f1_05': hist1.val_f1_05, 'best_epoch': hist1.best_epoch},
        'C2': {'train_loss': hist2.train_loss, 'val_mAP': hist2.val_mAP,
               'val_f1_05': hist2.val_f1_05, 'best_epoch': hist2.best_epoch},
        'C3': {'train_loss': hist3.train_loss, 'val_mAP': hist3.val_mAP,
               'val_f1_05': hist3.val_f1_05, 'best_epoch': hist3.best_epoch},
    },
    'thresholds_c4': {c: float(t) for c, t in zip(class_names, thresholds)},
    'winning_config': {
        'name': winner,
        'thresholds': [float(t) for t in thresholds]
                      if winner.startswith('C4')
                      else [0.5] * int(splits.C),
    },
    'test_report': test_report.to_dict(),
    'headline_test': {
        'macro_f1':    test_macro,
        'micro_f1':    test_micro,
        'weighted_f1': test_weighted,
    },
}

out_path = RESULTS / f'{NB_ID}_{MODEL_NAME}.json'
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2, default=float)
print(f'saved -> {out_path}')


saved -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results/08_cnn.json
